# KHDP API — Quickstart Notebook

[Open in Colab](https://colab.research.google.com/github/KoreaHealthDataPlatform/khdp-api/blob/main/examples/notebook/quickstart.ipynb)

A two-minute tour of the **Korea Health Data Platform** REST API through the `khdp` Python SDK.

- The **search** cells run without any credentials.
- The **download** cells need a personal API token (issued at <https://khdp.net> → *Settings → Account → API Token*).
- For end-to-end CLI / Claude Code / Codex examples, see the [repo README](https://github.com/KoreaHealthDataPlatform/khdp-api).

## Install

In [ ]:
%pip install -q khdp

## 1. Search public datasets (anonymous)

The dataset catalog accepts anonymous GETs. Only file listing and download require a credential.

In [ ]:
from khdp import Session

QUERY = "heart"   # change to whatever you're researching

with Session.open() as s:
    r = s.request("GET", "/datasets", params={"query": QUERY, "limit": 5})
r.raise_for_status()

items = r.json().get("items", [])
for d in items:
    print(f"{d['code']:<25} {d.get('title','')}")

## 2. Inspect a dataset

Pick the first search result and fetch its full metadata.

In [ ]:
import json

assert items, "No results — try a different QUERY in the cell above."
code = items[0]["code"]

with Session.open() as s:
    r = s.request("GET", f"/datasets/{code}/latest")
r.raise_for_status()
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:2000])

## 3. Authenticate for downloads

File listing and download require the `datasets` scope on a credential. The easiest credential for a notebook is a **personal API token** (`khdp_pat_…`):

1. Open <https://khdp.net> → *Settings → Account → API Token*.
2. Click **Issue** / **Regenerate**.
3. Paste the token below.

It's kept only in this kernel's memory; refresh the runtime to wipe it.

In [ ]:
import getpass, os

os.environ["KHDP_TOKEN"] = getpass.getpass("KHDP_TOKEN (khdp_pat_…): ").strip()

## 4. List files (authenticated)

In [ ]:
with Session.open() as s:
    r = s.authed_request("GET", f"/datasets/{code}/latest/files")

if r.status_code == 400 and "Open Access" in r.text:
    print(f"{code} is not an Open-policy dataset; pick a different one from cell 1.")
else:
    r.raise_for_status()
    for entry in r.json().get("items", [])[:20]:
        print(entry.get("key"), entry.get("size"))

## 5. Download a single file

We ask the API for a presigned URL, then fetch it directly. Presigned URLs carry the signature inline, so no auth header is needed for the actual download.

In [ ]:
import httpx
from pathlib import Path

# Set KEY to a file path from the listing above, e.g. "imaging/scan001.dcm".
KEY = None
assert KEY, "Set KEY to a file path from the previous cell's output."

with Session.open() as s:
    r = s.authed_request(
        "GET", f"/datasets/{code}/latest/files/{KEY}",
    )
r.raise_for_status()
url = r.json()["url"]

out = Path(Path(KEY).name)
with httpx.stream("GET", url) as resp:
    resp.raise_for_status()
    with out.open("wb") as f:
        for chunk in resp.iter_bytes():
            f.write(chunk)
print(f"Saved {out} ({out.stat().st_size} bytes)")

## Next steps

- **End-to-end walkthrough** — [`docs/quickstart.en.md`](https://github.com/KoreaHealthDataPlatform/khdp-api/blob/main/docs/quickstart.en.md)
- **API reference (Redoc)** — [https://khdp.io/docs](https://khdp.io/docs)
- **From an AI coding agent** — `claude mcp add khdp -- khdp-mcp`, then ask Claude Code to use the khdp tools.
- **Bulk download every file of a dataset** — [`examples/python/03_authenticated_download.py`](https://github.com/KoreaHealthDataPlatform/khdp-api/blob/main/examples/python/03_authenticated_download.py).